# VLM-DENTAL - YOLO Grounding Workspace

This notebook handles dataset downloading, YOLO grounding tool training with 5-fold cross-validation, and pushing generated artifacts to GitHub.


## 1. Environment Setup & Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**(Optional) Fresh Start Cleanup:**
Run this cell ONLY if you need to completely delete the VLM-DENTAL folder from your Google Drive to start over.

In [ ]:
# Uncomment the line below to delete the folder, then run the cell
# !rm -rf /content/drive/MyDrive/VLM-DENTAL

In [2]:
import os

# Set this to True to save the 10GB dataset and repo itself to Google Drive.
# Set this to False to keep the repo/dataset in temporary Colab storage.
SAVE_DATASET_AND_CODE_TO_DRIVE = False

# Set this to True to save generated YOLO model outputs to Google Drive.
# Set this to False to keep YOLO weights/results in the active repo clone under /content/.../data/models.
SAVE_YOLO_RESULTS_TO_DRIVE = False

drive_path = "/content/drive/MyDrive/VLM-DENTAL"
colab_path = "/content/VLM-DENTAL"
work_dir = drive_path if SAVE_DATASET_AND_CODE_TO_DRIVE else colab_path
models_root = f"{drive_path}/data/models" if SAVE_YOLO_RESULTS_TO_DRIVE else f"{work_dir}/data/models"
os.environ["YOLO_MODELS_ROOT"] = models_root

In [3]:
import os

if SAVE_DATASET_AND_CODE_TO_DRIVE:
    os.chdir("/content/drive/MyDrive")
else:
    os.chdir("/content")

if not os.path.exists("VLM-DENTAL"):
    os.system("git clone https://github.com/rezaxr14/VLM-DENTAL.git")

os.chdir(work_dir)
os.system("git pull")

# Ensure the selected output root exists without pulling everything into Drive by default.
os.makedirs(models_root, exist_ok=True)
if SAVE_YOLO_RESULTS_TO_DRIVE:
    os.makedirs(f"{drive_path}/data/traces", exist_ok=True)

In [4]:
# Install the project and all its requirements
!pip install -e .
!pip install python-dotenv pandas pillow google-generativeai anthropic huggingface_hub ultralytics

Obtaining file:///content/VLM-DENTAL
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of vllm to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 11.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of quack-kernels to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 37.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 22.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.6
    Uninstalling protobuf-6.33.6:
      Successfully uninstalled protobuf-6.33.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nvidia-cutlass-dsl-libs-cu13 4.6.0 requires protobuf<7,>=6.30.2, but you have protobuf 5.29.6 which is incompatible.
nvidia-cutlass-dsl-libs-cu12 4.6.0 requires protobuf<7,>=6.30.2, but you have protobuf 5.29.6 which is incompatible.
nvidia-cutlass-dsl-libs-core 4.6.0 requires protobuf<7,>=6.30.2, but you have protobuf 5.29.6 which is incompatible.
nvidia-cutlass-dsl-libs-base 4.6.0 requires protobuf<7,>=6.30.2, but you have protobuf 5.29.6 which is incompatible.


## 2. Configure Credentials (Colab Secrets Tab Support)
Loads API keys and GitHub tokens automatically from Colab Secrets tab (`google.colab.userdata`).

In [ ]:
!pip install -q python-dotenv

import os
from dotenv import load_dotenv

# Try to load secrets from /content/.env
env_path = "/content/.env"
if os.path.exists(env_path):
    print(f"Loading environment variables from {env_path}")
    load_dotenv(env_path)
else:
    print(f"Warning: {env_path} not found. Attempting to use Colab Secrets or placeholders.")

# 1. Attempt loading keys from Google Colab Secrets (userdata tab) as a fallback
try:
    from google.colab import userdata
    
    # Hugging Face Token
    try:
        hf_token = userdata.get('HF_TOKEN')
        if hf_token and 'HF_TOKEN' not in os.environ:
            os.environ['HF_TOKEN'] = hf_token
    except Exception:
        pass

    # GitHub Access Token
    try:
        gh_token = userdata.get('GITHUB_TOKEN') or userdata.get('GH_TOKEN')
        if gh_token and 'GITHUB_TOKEN' not in os.environ:
            os.environ['GITHUB_TOKEN'] = gh_token
            os.environ['GH_TOKEN'] = gh_token
    except Exception:
        pass

    # Git User Info
    try:
        git_name = userdata.get('GIT_USER_NAME')
        if git_name and 'GIT_USER_NAME' not in os.environ:
            os.environ['GIT_USER_NAME'] = git_name
    except Exception:
        pass

    try:
        git_email = userdata.get('GIT_USER_EMAIL')
        if git_email and 'GIT_USER_EMAIL' not in os.environ:
            os.environ['GIT_USER_EMAIL'] = git_email
    except Exception:
        pass

except ImportError:
    pass


if 'HF_TOKEN' not in os.environ or os.environ['HF_TOKEN'].startswith('YOUR_'):
    os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN_HERE'

if 'GITHUB_TOKEN' not in os.environ:
    os.environ['GITHUB_TOKEN'] = 'YOUR_GITHUB_TOKEN_HERE'

# Status Report
print("--- Credentials Status ---")
print(f"Hugging Face Token set: {'Yes' if os.environ.get('HF_TOKEN') and not os.environ['HF_TOKEN'].startswith('YOUR_') else 'No (using placeholder)'}")
print(f"GitHub Token set: {'Yes' if os.environ.get('GITHUB_TOKEN') and not os.environ['GITHUB_TOKEN'].startswith('YOUR_') else 'No (using placeholder)'}")


Loading environment variables from /content/.env
--- Credentials Status ---
Gemini API Key set: No (using placeholder)
Hugging Face Token set: Yes
GitHub Token set: Yes


## 3. Dataset Download & Cleanup
Run this to download the dataset if you haven't already. It will extract and structure it automatically.

In [5]:
!python download_and_cleanup.py

Using cache directory: /content/VLM-DENTAL/hf_cache

DENTEX/training_data.zip: downloading bytes:   4% 427M/10.9G [00:02<00:26, 393MB/s, 32.3MB/s  ]  
DENTEX/training_data.zip: downloading bytes:   6% 671M/10.9G [00:03<00:19, 513MB/s, 52.7MB/s  ]
DENTEX/training_data.zip: downloading bytes:   7% 762M/10.9G [00:03<00:20, 486MB/s, 63.6MB/s  ]  ]
DENTEX/training_data.zip: downloading bytes:   9% 1.03G/10.9G [00:04<00:24, 397MB/s, 83.2MB/s  ] ]
DENTEX/training_data.zip: downloading bytes:  18% 1.95G/10.9G [00:06<00:20, 444MB/s,  143MB/s  ]] 
DENTEX/training_data.zip: downloading bytes:  25% 2.74G/10.9G [00:08<00:35, 233MB/s,  179MB/s  ]]
DENTEX/training_data.zip: downloading bytes:  29% 3.20G/10.9G [00:11<00:32, 237MB/s,  183MB/s  ] ]
DENTEX/training_data.zip: downloading bytes:  31% 3.38G/10.9G [00:11<00:28, 267MB/s,  187MB/s  ] ]
DENTEX/training_data.zip: downloading bytes:  43% 4.70G/10.9G [00:15<00:17, 366MB/s,  234MB/s  ] ]
DENTEX/training_data.zip: downloading bytes:  50% 5.43G/10.9G

In [8]:
# Delete unused partial datasets to save space
!rm -rf data/dentex/DENTEX/training_data/disease
!rm -rf data/dentex/DENTEX/training_data/quadrant
!rm -rf data/dentex/DENTEX/training_data/unlabelled/

!rm -rf data/dentex/DENTEX/testing_data/disease
!rm -rf data/dentex/DENTEX/testing_data/quadrant

# Delete corrupted cache folder from any previous bugs (if it exists)
!rm -rf "C:\\Users\\rezax\\dental_agent_cache"

## 4b. Multi-Dataset Note (Scaffolded, Not Yet Wired Here)
The same `DATASET_NAME` convention used in the TraceGen notebook applies conceptually here
too, but this notebook's COCO-to-YOLO conversion (`scripts/prepare_yolo_dataset.py`, next
cell) is still DENTEX-only -- combining annotations from a second dataset into one
training run needs real Tufts bounding boxes to combine with, which don't exist yet (see
`dental_agent/data/tufts.py`'s module docstring: the tooth-position/diagnosis mapping is
intentionally not guessed at). Revisit `prepare_yolo_dataset.py` once that mapping lands
and generalizing `locate_tooth` across datasets becomes the actual next step.

## 5. YOLO Grounding Tool — 5-Fold Cross-Validation
Convert DENTEX annotations to YOLO format with 5-fold CV splits. Each fold trains independently from scratch, then the best fold's weights are selected as the final grounding model. The 50 validation images are held out permanently and never used in any fold's training.

In [6]:
# Convert COCO annotations to YOLO format with 5-fold cross-validation splits
# - Training pool (1339 images) is split into 5 folds via KFold
# - 50 validation images are held out permanently as a separate test set
!python scripts/prepare_yolo_dataset.py --mode cv --folds 5 

Preparing YOLO Dataset (5-fold cross-validation mode)...
Loading training split from DENTEX (combine_enumeration_splits=True)...
✅ Found existing DENTEX dataset for 'train' split at: data/dentex/DENTEX
Combined 2 annotation file(s) for 'train': 1339 images, 21624 annotations.
Training pool: 1339 images
Loading validation split from DENTEX (held-out test set)...
Held-out test set: 50 images

--- Fold 1/5 ---
Fold 0 train: 100% 1071/1071 [00:27<00:00, 38.86it/s]
Fold 0 val: 100% 268/268 [00:10<00:00, 25.90it/s]
Fold 0: 1049 train images, 263 val images

--- Fold 2/5 ---
Fold 1 train: 100% 1071/1071 [00:31<00:00, 33.77it/s]
Fold 1 val: 100% 268/268 [00:08<00:00, 32.49it/s]
Fold 1: 1052 train images, 260 val images

--- Fold 3/5 ---
Fold 2 train: 100% 1071/1071 [00:26<00:00, 40.24it/s]
Fold 2 val: 100% 268/268 [00:06<00:00, 40.66it/s]
Fold 2: 1049 train images, 263 val images

--- Fold 4/5 ---
Fold 3 train: 100% 1071/1071 [00:24<00:00, 44.07it/s]
Fold 3 val: 100% 268/268 [00:06<00:00, 40.6

In [14]:
# ==============================================================================
# YOLO TRAINING CELL (WITH AUTO-DOWNLOAD FROM HF)
# ==============================================================================
import os
import shutil
from huggingface_hub import snapshot_download

# Training config
RESUME = True  # Set True to continue training from last checkpoint (safe to leave True)
SYNC_FROM_HF = True  # Set True to download previous folds from HF before starting

if RESUME and SYNC_FROM_HF:
    hf_token = os.environ.get("HF_TOKEN")
    hf_repo = os.environ.get("HF_ARTIFACT_REPO", "Reza-Nadimi/vlm-dental-models")
    
    if hf_token and not hf_token.startswith("YOUR_"):
        print(f"Checking {hf_repo} for previous folds...")
        try:
            dl_dir = "/tmp/hf_download"
            snapshot_download(
                repo_id=hf_repo,
                repo_type="model",
                allow_patterns="yolo_cv/*",
                local_dir=dl_dir,
                local_dir_use_symlinks=False,
                token=hf_token
            )
            src_dir = os.path.join(dl_dir, "yolo_cv")
            if os.path.exists(src_dir):
                shutil.copytree(src_dir, models_root, dirs_exist_ok=True)
                print(f"Successfully restored models to {models_root}")
            else:
                print("No previous folds found on Hugging Face (repo is empty or missing yolo_cv folder).")
        except Exception as e:
            print(f"Failed to sync from HF: {e}")
    else:
        print("Warning: HF_TOKEN not set, skipping Hugging Face sync.")

# Build command — only pass --resume if there is a checkpoint to resume from
resume_flag = "--resume" if RESUME else ""
!YOLO_MODELS_ROOT="{models_root}" python scripts/train_grounding_tool.py --cross-validate --model yolov8m.pt --epochs 40 --patience 10 --batch 16 --imgsz 1024 --device 0 $resume_flag


Checking Reza-Nadimi/vlm-dental-models for previous folds...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 82 files:   0%|          | 0/82 [00:00<?, ?it/s]

Successfully restored models to /content/VLM-DENTAL/data/models

  FOLD 1/5 — already complete, skipping.

  FOLD 2/5 — already complete, skipping.

  FOLD 3/5 — already complete, skipping.

  FOLD 4/5 — already complete, skipping.

  FOLD 5/5
Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/yolo_dentex_cv/fold_4/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015

# MANUAL HUGGING FACE BACKUP CELL
# Run this cell after stopping training to safely back up your finished folds!


In [ ]:
import os
from huggingface_hub import HfApi

hf_token = os.environ.get("HF_TOKEN")
hf_repo = os.environ.get("HF_ARTIFACT_REPO", "Reza-Nadimi/vlm-dental-models")

if not hf_token or hf_token.startswith("YOUR_"):
    print("ERROR: HF_TOKEN is missing in environment!")
else:
    print(f"Uploading {models_root} to Hugging Face Hub: {hf_repo}/yolo_cv ...")
    api = HfApi(token=hf_token)
    try:
        api.create_repo(repo_id=hf_repo, repo_type="model", exist_ok=True)
        api.upload_folder(
            folder_path=models_root,
            path_in_repo="yolo_cv",
            repo_id=hf_repo,
            repo_type="model",
            commit_message="Backup YOLO CV folds"
        )
        print("Backup Complete! It is now safe to disconnect your Colab session.")
    except Exception as e:
        print(f"Upload failed: {e}")


# or run the following in the terminal without the !

In [ ]:
!tr -d '\r' < /content/.env > /content/.env_clean
!set -a; source /content/.env_clean; set +a
!hf upload Reza-Nadimi/vlm-dental-models /content/VLM-DENTAL/data/models yolo_cv --repo-type model --token $HF_TOKEN

# print results

In [ ]:
import json
from pathlib import Path

results_path = Path("data/models/grounding_tool_cv_best/cv_results.json")
if results_path.exists():
    results = json.loads(results_path.read_text())
    print(f"{'Fold':<6} {'mAP50':<10} {'mAP50-95':<10} {'Precision':<10} {'Recall':<10}")
    print("-" * 46)
    for r in results["folds"]:
        marker = " <-- BEST" if r["fold"] == results["best_fold"] else ""
        print(f"{r['fold']:<6} {r['map50']:<10.4f} {r['map50_95']:<10.4f} {r['precision']:<10.4f} {r['recall']:<10.4f}{marker}")
    print("-" * 46)
    print(f"Mean:  {results['mean_map50']:.4f} +/- {results['std_map50']:.4f}   {results['mean_map50_95']:.4f} +/- {results['std_map50_95']:.4f}")
    print(f"\nBest fold: {results['best_fold']}")
    print(f"Best weights: data/models/grounding_tool_cv_best/weights/best.pt")
else:
    print("CV results not found. Run the cross-validation training cell first.")

## 6. Download Generated Models Locally
Triggers interactive browser download of trained YOLO weights (`best.pt`).

In [ ]:
import os
import shutil

# Options
DOWNLOAD_CV_BEST = True       # Best fold weights (final model)
DOWNLOAD_ALL_FOLDS = False    # Set True to download all 5 fold weights

files_to_download = []
cv_best_pt = f"{drive_path}/data/models/grounding_tool_cv_best/weights/best.pt" if SAVE_YOLO_RESULTS_TO_DRIVE else f"{work_dir}/data/models/grounding_tool_cv_best/weights/best.pt"
cv_results_json = f"{drive_path}/data/models/grounding_tool_cv_best/cv_results.json" if SAVE_YOLO_RESULTS_TO_DRIVE else f"{work_dir}/data/models/grounding_tool_cv_best/cv_results.json"

# Prepare CV Best Weights
if DOWNLOAD_CV_BEST and os.path.exists(cv_best_pt):
    files_to_download.append(cv_best_pt)
elif DOWNLOAD_CV_BEST:
    print(f"CV best weights not found at: {cv_best_pt}")

# Prepare CV Results JSON
if DOWNLOAD_CV_BEST and os.path.exists(cv_results_json):
    files_to_download.append(cv_results_json)

# Prepare All Fold Weights
if DOWNLOAD_ALL_FOLDS:
    for fold in range(5):
        fold_pt = f"{drive_path}/data/models/cv_fold_{fold}/weights/best.pt" if SAVE_YOLO_RESULTS_TO_DRIVE else f"{work_dir}/data/models/cv_fold_{fold}/weights/best.pt"
        if os.path.exists(fold_pt):
            files_to_download.append(fold_pt)
        else:
            print(f"Fold {fold} weights not found at: {fold_pt}")

# Download via Colab files API
try:
    from google.colab import files
    for file_path in files_to_download:
        print(f"Triggering download for: {file_path}")
        files.download(file_path)
except ImportError:
    print("Colab files utility not available. Files located at:")
    for f in files_to_download:
        print(" -", f)


## 7. Push Models to GitHub
Configures git authentication using `GITHUB_TOKEN` from secrets and pushes newly generated models to GitHub.

In [ ]:
import os
import shutil
import subprocess
import json

# Options
PUSH_MODELS = True  # Set to True if model weights are within Git repository size limits
COMMIT_MESSAGE = "Update trained YOLO grounding models & CV results"

gh_token = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')

if not gh_token or gh_token.startswith('YOUR_'):
    print("ERROR: GITHUB_TOKEN is not set. Add GITHUB_TOKEN to your Colab Secrets tab or environment variables.")
else:
    git_user = os.environ.get('GIT_USER_NAME', 'Reza Nadimi')
    git_email = os.environ.get('GIT_USER_EMAIL', 'rezaxr14@gmail.com')

    # Configure Git credentials
    subprocess.run(["git", "config", "--global", "user.name", git_user])
    subprocess.run(["git", "config", "--global", "user.email", git_email])

    # Set Remote URL with Token
    remote_url = f"https://{gh_token}@github.com/rezaxr14/VLM-DENTAL.git"
    subprocess.run(["git", "remote", "set-url", "origin", remote_url])

    if PUSH_MODELS:
        best_dir = f"{drive_path}/data/models/grounding_tool_cv_best" if SAVE_YOLO_RESULTS_TO_DRIVE else f"{work_dir}/data/models/grounding_tool_cv_best"
        
        if os.path.exists(best_dir):
            # 1. Stage the models and images
            subprocess.run(["git", "rm", "-r", "--cached", "data/models/"], check=False)
            subprocess.run(["git", "add", "data/models/grounding_tool_cv_best/"])
            
            # 2. Generate the Markdown Results file from the JSON
            cv_json = os.path.join(best_dir, "cv_results.json")
            if os.path.exists(cv_json):
                with open(cv_json, 'r') as f:
                    metrics = json.load(f)
                
                md = "# YOLO Grounding Tool - Cross-Validation Results\n\n"
                md += "| Fold | mAP50 | mAP50-95 | Precision | Recall |\n"
                md += "|------|-------|----------|-----------|--------|\n"
                for r in metrics['folds']:
                    marker = " ⭐ (BEST)" if r['fold'] == metrics['best_fold'] else ""
                    md += f"| {r['fold']}{marker} | {r['map50']:.4f} | {r['map50_95']:.4f} | {r['precision']:.4f} | {r['recall']:.4f} |\n"
                
                md += f"\n**Mean mAP50:** {metrics['mean_map50']:.4f} ± {metrics['std_map50']:.4f}\n"
                md += f"**Mean mAP50-95:** {metrics['mean_map50_95']:.4f} ± {metrics['std_map50_95']:.4f}\n"
                
                md_path = f"{work_dir}/YOLO_CV_RESULTS.md"
                with open(md_path, 'w') as f:
                    f.write(md)
                
                # Add the new markdown file to git!
                subprocess.run(["git", "add", "YOLO_CV_RESULTS.md"])

    # Check status and commit
    status_res = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True)
    if status_res.stdout.strip():
        subprocess.run(["git", "commit", "-m", COMMIT_MESSAGE])
        print("Pushing changes to GitHub repository...")
        subprocess.run(["git", "push", "origin", "main"])
        print("GitHub Push Complete!")
    else:
        print("No new changes detected to commit.")


In [ ]:
!git pull

Already up to date.
